#**Instalación de Dependencias y validación de columnas**

In [ ]:
# 1. Instalar dependencias necesarias
!pip install psycopg2-binary sqlalchemy pandas

import pandas as pd
from sqlalchemy import create_engine, text
from google.colab import userdata
userdata.get('NEON_URL')
# Assign the actual Neon URL to DATABASE_URL
DATABASE_URL = userdata.get('NEON_URL')
# Tu connection string de Neon
engine = create_engine(DATABASE_URL)

# 2. Cargar y Validar el Dataset
df = pd.read_csv('manufacturing_defect_dataset.csv')

print("--- VALIDACIÓN INICIAL DEL DATASET ---")
print(f"Dimensiones (shape): {df.shape}")
print("\nTipos de datos (dtypes):")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum())

# Crear columna extra 'id_proceso' (index + 1)
df.insert(0, 'id_proceso', df.index + 1)

print("\n--- VISTA PREVIA CON ID_PROCESO ---")
display(df.head())

--- VALIDACIÓN INICIAL DEL DATASET ---
Dimensiones (shape): (3240, 17)

Tipos de datos (dtypes):
ProductionVolume          int64
ProductionCost          float64
SupplierQuality         float64
DeliveryDelay             int64
DefectRate              float64
QualityScore            float64
MaintenanceHours          int64
DowntimePercentage      float64
InventoryTurnover       float64
StockoutRate            float64
WorkerProductivity      float64
SafetyIncidents           int64
EnergyConsumption       float64
EnergyEfficiency        float64
AdditiveProcessTime     float64
AdditiveMaterialCost    float64
DefectStatus              int64
dtype: object

Valores nulos por columna:
ProductionVolume        0
ProductionCost          0
SupplierQuality         0
DeliveryDelay           0
DefectRate              0
QualityScore            0
MaintenanceHours        0
DowntimePercentage      0
InventoryTurnover       0
StockoutRate            0
WorkerProductivity      0
SafetyIncidents         0
Energ

,id_proceso,ProductionVolume,ProductionCost,SupplierQuality,DeliveryDelay,DefectRate,QualityScore,MaintenanceHours,DowntimePercentage,InventoryTurnover,StockoutRate,WorkerProductivity,SafetyIncidents,EnergyConsumption,EnergyEfficiency,AdditiveProcessTime,AdditiveMaterialCost,DefectStatus
0,1,202,13175.403783,86.648534,1,3.121492,63.463494,9,0.052343,8.630515,0.081322,85.042379,0,2419.616785,0.468947,5.551639,236.439301,1
1,2,535,19770.046093,86.310664,4,0.819531,83.697818,20,4.908328,9.296598,0.038486,99.657443,7,3915.566713,0.119485,9.080754,353.957631,1
2,3,960,19060.820997,82.132472,0,4.514504,90.350550,1,2.464923,5.097486,0.002887,92.819264,2,3392.385362,0.496392,6.562827,396.189402,1
3,4,370,5647.606037,87.335966,5,0.638524,67.628690,8,4.692476,3.577616,0.055331,96.887013,8,4652.400275,0.183125,8.097496,164.135870,1
4,5,206,7472.222236,81.989893,3,3.867784,82.728334,9,2.746726,6.851709,0.068047,88.315554,7,1581.630332,0.263507,6.406154,365.708964,1


In [ ]:
try:
    # 3. Cargar el DataFrame a la tabla 'manufacturing'
    # Usamos if_exists='replace' para sobreescribir si ya existe
    df.to_sql('manufacturing', engine, if_exists='replace', index=False)
    print("Datos cargados exitosamente en la tabla 'manufacturing'.")
except Exception as e:
    print(f"Error al cargar datos: {e}")

# 4. Función query(sql) para retornar DataFrames
def query(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

# 5. Verificar conexión
print("\n--- VERIFICACIÓN DE CONEXIÓN ---")
version = query("SELECT version();")
print(f"Conectado a: {version.iloc[0,0]}")

Datos cargados exitosamente en la tabla 'manufacturing'.

--- VERIFICACIÓN DE CONEXIÓN ---
Conectado a: PostgreSQL 17.8 (a284a84) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


# **1. Percentil**

In [ ]:
#  Percentil 90 de DefectRate
sql_percentil = """
SELECT
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY "DefectRate")
    as p90_defectos,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY "DefectRate")
    as mediana_defectos
FROM manufacturing;
"""

df_resultado = query(sql_percentil)
display(df_resultado)

,p90_defectos,mediana_defectos
0,4.572254,2.708775


# **2. Filter**

In [ ]:
#  Agregación condicional elegante
sql_filter = """
SELECT
    COUNT(*) FILTER (WHERE "DefectStatus" = 1) as defectuosos,
    COUNT(*) FILTER (WHERE "DefectStatus" = 0) as conformes,
    ROUND(CAST(AVG("DefectRate") FILTER (WHERE "DefectStatus" = 1) AS NUMERIC), 4)
        as avg_defect_rate_criticos
FROM manufacturing;
"""

df_resultado = query(sql_filter)
display(df_resultado)

,defectuosos,conformes,avg_defect_rate_criticos
0,2723,517,2.8894


# **3. NTILE**

In [17]:
#  Clasificar  procesos en 4 cuartiles de calidad
sql_filter = """
SELECT
    id_proceso,
    "QualityScore",
    "DefectRate",
    NTILE(4) OVER (ORDER BY "QualityScore" DESC) as cuartil_calidad
FROM manufacturing;
"""
df_percentil = query(sql_filter)
display(df_percentil)

,id_proceso,QualityScore,DefectRate,cuartil_calidad
0,1853,99.996993,2.872846,1
1,1205,99.992206,3.695653,1
2,1380,99.978363,3.903493,1
3,2907,99.965983,1.809515,1
4,2842,99.945915,4.306916,1
...,...,...,...,...
3235,1056,60.052795,1.628293,4
3236,1505,60.022167,2.760793,4
3237,2236,60.013206,3.292516,4
3238,1949,60.012389,3.566815,4


# **4. Window Frame Promedio móvil**

In [ ]:
#  Análisis de tendencia
sql_pm = """
SELECT
    id_proceso,
    "ProductionVolume",
    ROUND(AVG("ProductionVolume") OVER (
        ORDER BY id_proceso
        ROWS BETWEEN 2 PRECEDING AND 2 FOLLOWING
    ), 2) as promedio_movil_5
FROM manufacturing
ORDER BY id_proceso;
"""
df_resultado = query(sql_pm)
display(df_resultado)

,id_proceso,ProductionVolume,promedio_movil_5
0,1,202,565.67
1,2,535,516.75
2,3,960,454.60
3,4,370,448.40
4,5,206,501.40
...,...,...,...
3235,3236,762,603.00
3236,3237,335,596.00
3237,3238,835,517.80
3238,3239,302,456.75


# **Preguntas de negocio**

### **Nivel 1 — Distribución estadística:**


- ¿Cuál es la mediana y el percentil 90 de DefectRate?

**Respuesta:** La mediana y el percentil es de  2.70 y 4.57 respectivamente, lo que indica que el 50% de los datos son menores de 2.70 y mayores de 2.70 defectos, y el 90% de los datos tienen menos de 5 defectos.
- ¿En qué se diferencia la mediana del promedio que calculaste el Día 10?

**Respuesta:**
Mediana (2.70): La mitad de tus datos es menor o igual a 2.70, y la otra mitad es mayor. Es el valor central real.
Promedio (2.74): Es la suma de todo dividido por la cantidad de datos. La cercanía indica una base operativa relativamente estable. Sin embargo, el salto drástico hacia el **Percentil 90 (4.57)** confirma que el problema de rentabilidad no es una falla de toda la línea, sino la presencia de **outliers críticos**. No necesitamos reestructurar toda la planta; necesitamos identificar qué sucede específicamente en ese 10% de procesos que están operando al borde del colapso técnico.




### **Nivel 2 — Segmentación operacional:**



- ¿Cuántos procesos caen en el cuartil crítico (cuartil 4 — peor calidad)?

**Respuesta:** Hay 810 procesos en el cuartil crítico, información que invita a realizar el Control Estadistico de la Calidad y los procesos, para determinar las causas de porque están fallando los procesos.
- Promedio móvil de ProductionVolume para detectar


In [19]:
# Contar cuántos procesos están en el cuartil 4 (peor calidad)
peores_procesos = df_percentil[df_percentil['cuartil_calidad'] == 4]
cantidad_critica = len(peores_procesos)

print(f"Respuesta de Negocio: Hay {cantidad_critica} procesos en el cuartil crítico.")

Respuesta de Negocio: Hay 810 procesos en el cuartil crítico.


**Nivel 3 — Análisis ejecutivo:**
- Combina NTILE + CTE: del cuartil 4 (peor calidad), ¿cuál es el promedio de WorkerProductivity y MaintenanceHours?

**Respuesta:** La productividad promedio es de 90.22% con un mantenimiento promedio de 11.51 horas.

- ¿Los procesos con más horas de mantenimiento tienen menor DefectRate? (correlación descriptiva con AVG por segmento)

**Respuesta:** Los procesos con más horas de mantenimiento no está correlacioando con la tasa de defectos, puesto que ofrece una tasa promedio de 2.73 de 2.76 de los que tienen menos horas de mantenimiento

In [ ]:
# SQL Avanzado: Filtrando el Cuartil Crítico para análisis de recursos
sql_ejecutivo = """
WITH ClasificacionCalidad AS (
    SELECT
        id_proceso,
        "QualityScore",
        "WorkerProductivity",
        "MaintenanceHours",
        NTILE(4) OVER (ORDER BY "QualityScore" DESC) as cuartil_calidad
    FROM manufacturing
)
SELECT
    COUNT(id_proceso) as total_procesos,
    ROUND(AVG("WorkerProductivity")::numeric, 2) as avg_productividad,
    ROUND(AVG("MaintenanceHours")::numeric, 2) as avg_mantenimiento
FROM ClasificacionCalidad
WHERE cuartil_calidad = 4;
"""

df_ejecutivo = query(sql_ejecutivo)
display(df_ejecutivo)

,total_procesos,avg_productividad,avg_mantenimiento
0,810,90.22,11.51


In [ ]:
# SQL Avanzado: Relación Mantenimiento vs. Defectos
sql_correlacion = """
WITH SegmentacionMantenimiento AS (
    SELECT
        "DefectRate",
        "MaintenanceHours",
        CASE
            WHEN "MaintenanceHours" < 20 THEN '1. Bajo (<20h)'
            WHEN "MaintenanceHours" BETWEEN 20 AND 50 THEN '2. Medio (20-50h)'
            ELSE '3. Alto (>50h)'
        END AS nivel_mantenimiento
    FROM manufacturing
)
SELECT
    nivel_mantenimiento,
    COUNT(*) as total_procesos,
    ROUND(AVG("DefectRate")::numeric, 4) as avg_defect_rate
FROM SegmentacionMantenimiento
GROUP BY nivel_mantenimiento
ORDER BY nivel_mantenimiento;
"""

df_correlacion = query(sql_correlacion)
display(df_correlacion)

,nivel_mantenimiento,total_procesos,avg_defect_rate
0,1. Bajo (<20h),2733,2.7516
1,2. Medio (20-50h),507,2.7355


# Exportar Archivo SQL

In [ ]:
# Definición de las consultas del día
queries = {
    "1_verificacion_conexion": "SELECT version();",
    "2_cuartil_critico_conteo": """
        SELECT cuartil_calidad, COUNT(id_proceso) as total_procesos
        FROM (
            SELECT id_proceso, NTILE(4) OVER (ORDER BY "QualityScore" DESC) as cuartil_calidad
            FROM manufacturing
        ) as subconsulta
        WHERE cuartil_calidad = 4
        GROUP BY cuartil_calidad;
    """,
    "3_analisis_ejecutivo_causa_raiz": """
        WITH ClasificacionCalidad AS (
            SELECT id_proceso, "QualityScore", "WorkerProductivity", "MaintenanceHours",
                   NTILE(4) OVER (ORDER BY "QualityScore" DESC) as cuartil_calidad
            FROM manufacturing
        )
        SELECT COUNT(id_proceso) as total, ROUND(AVG("WorkerProductivity")::numeric, 2) as avg_prod,
               ROUND(AVG("MaintenanceHours")::numeric, 2) as avg_maint
        FROM ClasificacionCalidad WHERE cuartil_calidad = 4;
    """,
    "4_correlacion_mantenimiento_defectos": """
        WITH Segmentacion AS (
            SELECT "DefectRate", "MaintenanceHours",
            CASE WHEN "MaintenanceHours" < 20 THEN '1. Bajo'
                 WHEN "MaintenanceHours" BETWEEN 20 AND 50 THEN '2. Medio'
                 ELSE '3. Alto' END AS nivel_mantenimiento
            FROM manufacturing
        )
        SELECT nivel_mantenimiento, ROUND(AVG("DefectRate")::numeric, 4) as avg_defect_rate
        FROM Segmentacion GROUP BY nivel_mantenimiento ORDER BY nivel_mantenimiento;
    """
}

# Exportar a archivo .sql
with open('queries_postgresql.sql', 'w') as f:
    for nombre, sql in queries.items():
        f.write(f"-- {nombre.replace('_', ' ').upper()}\n")
        f.write(sql.strip() + ";\n\n")

print("Archivo 'queries_postgresql.sql' generado con éxito.")

Archivo 'queries_postgresql.sql' generado con éxito.
